# 04 — Map every poultry house in one county, and check it

**Question.** If we run the detector over a *whole county*, not only around registered farms, how many poultry houses does it find, how accurate is it against houses traced by hand, and how does the county total compare with the registry and the AWTF report?

**How.** A grid of 2 km aerial-photo tiles over the county's land, overlapping by 250 m so a house is whole in at least one tile → the same U-Net + shape rule as notebooks 01–03 → overlapping copies merged (keeping the most complete outline) → each building's distance to the nearest registered permit.

**Checked against.**
1. **USGS hand-traced houses** (Soroka & Duren, 2016/17). They traced *every* poultry house on Delmarva, registered or not, so they grade a registry-independent run directly.
2. **By eye**: galleries of the disagreements and of buildings far from any permit.
3. **County totals**: MDE permits and the AWTF report's 2017 bird inventory (Table 1a.3).

**Inputs.** Census county outlines (downloaded once, cached in `data/reference/`), NAIP tiles from Planetary Computer, the U-Net checkpoint, the registry manifest.
**Outputs.** `data/processed/task1/county_<name>/` (same files as a registry run, plus `county_summary.json`).
**Runtime.** Downloading ~350–600 tiles takes about an hour per county; detection about 5 minutes. With `RUN_FULL = False` the notebook loads the last saved run in seconds.

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
# reload edited geo_anom/ code automatically, without restarting the kernel
%load_ext autoreload
%autoreload 2
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)

from geo_anom.task1 import tiles, region, viz
from geo_anom.task1.pipeline import run_pipeline
from geo_anom.task1.evaluate import evaluate_run, load_ground_truth

## Settings

In [ ]:
COUNTY = "Somerset"     # any Maryland county name, e.g. "Wicomico", "Queen Anne's"
RUN_FULL = False        # True: download tiles + run detection (~1 h); False: load the saved run
MAX_WORKERS = 8         # parallel tile downloads
NEAR_PERMIT_M = 1000    # a building within this distance of a registered permit counts as "near a permit"

slug = COUNTY.lower().replace(" ", "_").replace("'", "")
OUT = ROOT / f"data/processed/task1/county_{slug}"
print("results folder:", OUT.relative_to(ROOT))

## 1. The county and its tile grid

Each square is one 2 km × 2 km aerial photo. Squares overlap by 250 m. Squares that are almost all water are skipped. Dots are the registered CAFO/AFO permits inside the county.

In [ ]:
county = region.county_boundary(COUNTY)
sites = region.grid_sites(county, COUNTY)
registry = tiles.load_manifest()
reg = gpd.GeoDataFrame(registry, geometry=gpd.points_from_xy([s["lon"] for s in registry], [s["lat"] for s in registry]), crs=4326)
reg_in = reg[reg.within(county.unary_union)]

squares = gpd.GeoSeries([region.box(*gpd.GeoSeries(gpd.points_from_xy([s["lon"]], [s["lat"]]), crs=4326)
                          .to_crs(region.UTM).buffer(region.TILE_M / 2, cap_style=3).total_bounds) for s in sites],
                        crs=region.UTM)
fig, ax = plt.subplots(figsize=(9, 9))
county.to_crs(region.UTM).plot(ax=ax, color="#eef4ee", edgecolor="#2c5f2d", lw=1.5)
squares.plot(ax=ax, facecolor="none", edgecolor="#1c7293", lw=0.3)
reg_in.to_crs(region.UTM).plot(ax=ax, color="#c0392b", markersize=12, label="registered permit")
ax.set_axis_off(); ax.legend(loc="lower left")
ax.set_title(f"{COUNTY} County: {len(sites)} tiles, {len(reg_in)} registered permits")
land_km2 = county.ALAND.iloc[0] / 1e6
display(Markdown(f"**{len(sites)} tiles** over {land_km2:,.0f} km² of land · about {len(sites) * 10 / 1000:.1f} GB to download · "
                 f"**{len(reg_in)} registered permits** in the county ({reg_in.animal_type.value_counts().to_dict()})"))

## 2. Download the photos and run the detector

Only runs with `RUN_FULL = True`. Tiles already on disk are skipped, so an interrupted run can simply be restarted. Detection is identical to the registry run, except that when two tiles see the same barn, the **larger** (more complete) outline is kept, since a barn cut by one tile's edge is whole in its neighbour.

In [ ]:
if RUN_FULL:
    from tqdm.auto import tqdm
    ready = region.download_many(sites, max_workers=MAX_WORKERS, progress=tqdm)
    meta = run_pipeline(ready, OUT, manifest=registry, prefer="area",
                        filters={"farm": [], "county": [COUNTY]},
                        progress=lambda xs: tqdm(xs, desc="tiles", unit="tile"))
else:
    assert (OUT / "run_meta.json").exists(), f"No saved run in {OUT}; set RUN_FULL = True"
meta = json.loads((OUT / "run_meta.json").read_text())
sites_run = json.loads((OUT / "sites.json").read_text())
buildings = gpd.read_file(OUT / "buildings.geojson")
print(f"run {meta['run_at']} · code {meta['git_commit']} · {meta['tiles']} tiles · "
      f"{meta['kept_detections']:,} kept detections -> {meta['unique_buildings']:,} unique buildings")

## 3. Every building found, by distance to a registered permit

Green: within 1 km of a registered permit. Purple: farther. A purple building is not a finding by itself: it could be an operation below the CAFO threshold (legal, simply not in the registry), a registry point placed far from its barns, or a building that isn't a poultry house. Section 6 checks a sample by eye.

In [ ]:
b = buildings.to_crs(region.UTM)
b = b[b.centroid.within(county.to_crs(region.UTM).unary_union)].copy()   # this county only
b["near_permit"] = b["assigned_distance_m"] <= NEAR_PERMIT_M
fig, ax = plt.subplots(figsize=(9, 9))
county.to_crs(region.UTM).plot(ax=ax, color="#f4f6f4", edgecolor="#2c5f2d", lw=1.5)
b[b.near_permit].centroid.plot(ax=ax, color="#2c8c3c", markersize=5, label=f"within {NEAR_PERMIT_M} m of a permit ({b.near_permit.sum():,})")
b[~b.near_permit].centroid.plot(ax=ax, color="#8e44ad", markersize=7, label=f"farther ({(~b.near_permit).sum():,})")
reg_in.to_crs(region.UTM).plot(ax=ax, color="none", edgecolor="#c0392b", markersize=25, label="registered permit")
ax.set_axis_off(); ax.legend(loc="lower left")
ax.set_title(f"{COUNTY}: {len(b):,} buildings detected");

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4))
b.length_m.plot.hist(bins=40, ax=axs[0], color="#2c8c3c"); axs[0].set_title("Building length (m)")
b.assigned_distance_m.clip(upper=5000).plot.hist(bins=50, ax=axs[1], color="#8e44ad")
axs[1].axvline(NEAR_PERMIT_M, color="k", ls="--"); axs[1].set_title("Distance to nearest registered permit (m, capped at 5 km)")
plt.tight_layout()

## 4. Accuracy against houses traced by hand

**Precision**: of the buildings we mark, the share that overlap a hand-traced house. **Recall**: of the hand-traced houses, the share we overlap. Counted inside the county only.

Two caveats travel with these numbers: the hand tracing is from 2016/17 and the photos from 2023 (houses built or removed since then count as errors), and the shape rule was originally tuned on this same dataset.

In [ ]:
gt = load_ground_truth()
acc = evaluate_run(OUT, gt=gt, area=county)
o = acc["object_level"]
summary = pd.Series({
    "buildings detected (county, in hand-traced area)": o["detections"],
    "hand-traced houses (same area)": o["ground_truth_houses"],
    "precision": f"{o['precision']:.0%}",
    "recall": f"{o['recall']:.0%}",
    "F1": f"{o['f1']:.2f}",
    "median overlap (IoU) of matches": o["median_iou_of_matches"],
    "median distance to nearest hand-traced house (m)": acc["centroid_distance_m"]["median"],
}, name=COUNTY)
display(summary.to_frame())

In [ ]:
gtc = gt.to_crs(region.UTM)
gtc = gtc[gtc.centroid.within(county.to_crs(region.UTM).unary_union)]
hit_b = gpd.sjoin(b[["geometry"]], gtc[["geometry"]], predicate="intersects").index.unique()
hit_g = gpd.sjoin(gtc[["geometry"]], b[["geometry"]], predicate="intersects").index.unique()
false_pos = b[~b.index.isin(hit_b)]          # ours, no hand-traced house under it
missed = gtc[~gtc.index.isin(hit_g)]         # hand-traced, we found nothing
fig, ax = plt.subplots(figsize=(9, 9))
county.to_crs(region.UTM).plot(ax=ax, color="#f4f6f4", edgecolor="#2c5f2d", lw=1.5)
b[b.index.isin(hit_b)].centroid.plot(ax=ax, color="#2c8c3c", markersize=4, label=f"match ({len(hit_b):,})")
false_pos.centroid.plot(ax=ax, color="#e67e22", markersize=10, label=f"ours only ({len(false_pos):,})")
missed.centroid.plot(ax=ax, color="#c0392b", marker="x", markersize=14, label=f"hand-traced only ({len(missed):,})")
ax.set_axis_off(); ax.legend(loc="lower left"); ax.set_title("Where we agree and disagree with the hand tracing");

## 5. Look at the disagreements

Green outline = our building, orange = hand-traced house, yellow + = the location being shown. For each crop decide: real house we found that was built after 2017? Something that isn't a poultry house? A house we missed, or one that's gone?

In [ ]:
N = 12
rng = np.random.default_rng(0)
pick = lambda g: g.iloc[rng.choice(len(g), size=min(N, len(g)), replace=False)] if len(g) else g
fp = pick(false_pos)
viz.location_gallery(fp.centroid, sites_run, buildings=b, ground_truth=gtc,
                     titles=[f"ours only · {r.length_m:.0f}×{r.width_m:.0f} m" for _, r in fp.iterrows()],
                     suptitle=f"Ours only: {len(fp)} of {len(false_pos):,} (random sample)");

In [ ]:
mi = pick(missed)
viz.location_gallery(mi.centroid, sites_run, buildings=b, ground_truth=gtc,
                     titles=["hand-traced only"] * len(mi),
                     suptitle=f"Hand-traced only (we missed): {len(mi)} of {len(missed):,} (random sample)");

## 6. Buildings far from any registered permit

Possible reasons, all ordinary: an operation below the CAFO threshold, a permit point placed far from its barns, a recently built farm, or a building that isn't a poultry house. Look before counting any of them.

In [ ]:
far = b[~b.near_permit].sort_values("assigned_distance_m", ascending=False)
display(Markdown(f"**{len(far):,} buildings** are more than {NEAR_PERMIT_M} m from a registered permit; "
                 f"{far.index.isin(hit_b).sum():,} of them overlap a hand-traced house."))
sample = pick(far)
viz.location_gallery(sample.centroid, sites_run, buildings=b, ground_truth=gtc,
                     titles=[f"{r.assigned_distance_m / 1000:.1f} km from a permit" for _, r in sample.iterrows()],
                     suptitle="Buildings far from any registered permit (random sample)");

## 7. County totals

Counts of houses only. Converting houses to birds and then to nitrogen and phosphorus waits for the Extension numbers (birds per house, litter per house); see `docs/meeting_notes_2026-10-07_workshop.md`.

In [ ]:
awtf = pd.read_csv(ROOT / "data/reference/awtf_2017_broiler_layer_inventory_by_county.csv", comment="#").set_index("county")
row = awtf.loc[COUNTY] if COUNTY in awtf.index else pd.Series(dtype=float)
poultry_permits = reg_in[reg_in.animal_type.isin(["chickens_not_laying_hens", "laying_hens_dry_manure", "turkeys", "ducks_liquid_manure"])]
totals = pd.Series({
    "poultry houses detected (this notebook)": len(b),
    "  of which within 1 km of a registered permit": int(b.near_permit.sum()),
    "  of which farther": int((~b.near_permit).sum()),
    "hand-traced houses, 2016/17 (USGS)": len(gtc),
    "registered poultry permits (MDE)": len(poultry_permits),
    "permitted poultry headcount (MDE registry)": int(poultry_permits.headcount.sum()),
    "AWTF 2017 inventory, USDA Ag Census (birds)": row.get("ag_census_2017_birds"),
    "AWTF 2017 inventory, MDE CAFO data (birds)": row.get("mde_cafo_2017_birds"),
}, name=COUNTY)
display(totals.to_frame())
summary_out = {"county": COUNTY, "run": meta, "accuracy": acc, "totals": {k.strip(): (None if pd.isna(v) else float(v)) for k, v in totals.items()}}
(OUT / "county_summary.json").write_text(json.dumps(summary_out, indent=2, default=str))
print("saved", (OUT / "county_summary.json").relative_to(ROOT))

## What this shows

*Somerset County, first run (2026-10-10). Re-check these sentences if you re-run with different settings or another county.*

- **The grid finds more than the registry run, and almost nothing false.** 389 poultry houses in the county. Against the USGS hand tracing: **precision 99%, recall 66%**, outlines within ~4 m of the traced houses. The grid found **384** traced houses where the registry-centred run found 308 (+79, only 3 lost).
- **The 5 "ours only" buildings are real houses built after the 2016/17 tracing**, so true precision is effectively 100%.
- **81 houses are more than 1 km from any registered permit, and all 81 sit on a hand-traced house.** The 12 sampled are ordinary poultry farms 1–5 km from the nearest permit. They could be operations below the CAFO threshold or permits whose points are far from their barns. Either way, a registry-only map leaves out about a fifth of Somerset's houses.
- **The misses (196) are real houses, mostly older ones with dark or rust-coloured roofs.** Missed roofs average a brightness of 145 against 175 for found ones and lean red rather than blue; 65% are darker than the darkest tenth of the houses we find. The model learned bright metal roofs. This is the biggest gap and a specific one to work on next, e.g. retraining with Delmarva's dark-roof examples or a second, lower probability threshold for long, house-shaped shapes.
- **Totals:** 389 detected (≈ 2/3 of the 580 traced in 2016/17); 64 registered poultry permits with 8.8 M permitted birds; the AWTF report lists 10.1 M (MDE) and 12.5 M (USDA Census) birds for 2017. Bird and nutrient estimates wait for the Extension numbers.